# 🌾 Riftronix AgriTech — Amharic Agricultural Chatbot Fine-Tuning
### Production-Grade Notebook | T4 GPU | Unsloth + QLoRA + TRL

---

**Organization:** Riftronix Technology  by yeabsra Andnet 
**Role:** AI/ML Intern — LLM Fine-Tuning  
**Goal:** Fine-tune an open-source LLM on ~53,000 Amharic agricultural conversation pairs  
**Stack:** Unsloth · QLoRA · TRL · PEFT · Transformers · Accelerate  
**GPU:** Google Colab T4 (16 GB VRAM)  

---

## 📋 Notebook Sections

| # | Section | Description |
|---|---------|-------------|
| 1 | Environment Setup | Install all dependencies |
| 2 | Google Drive & Sheets | Mount Drive, load dataset |
| 3 | Data Validation | Schema checks, language detection |
| 4 | Data Cleaning | Normalize, deduplicate, filter |
| 5 | Tokenizer Analysis | Measure Amharic fertility, decide on extension |
| 6 | Dataset Formatting | Chat template, train/val/test split |
| 7 | Model Loading | Unsloth 4-bit quantized load |
| 8 | QLoRA Configuration | LoRA adapters, PEFT setup |
| 9 | Trainer Configuration | SFTTrainer, callbacks, scheduler |
| 10 | Fine-Tuning | Training loop with live metrics |
| 11 | Evaluation | Automated metrics + human-style scoring |
| 12 | Benchmarking | Cross-model & cross-config comparison |
| 13 | Save & Export | Adapters, merged model, GGUF |
| 14 | Troubleshooting | OOM, NaN loss, slow training fixes |

---

### ⚠️ Before You Start
1. Runtime → Change runtime type → **T4 GPU**
2. Ensure Google Drive has at least **30 GB free**
3. Keep the tab active or use Colab Pro to avoid disconnection
4. Set your Google Sheet ID in Section 2

---
## 📦 SECTION 1 — Environment Setup & Dependency Installation

### Why these packages?
- **unsloth**: 2× faster fine-tuning, 60% less VRAM via fused kernels — essential on T4
- **trl (SFTTrainer)**: Production-grade supervised fine-tuning with packing, masking, and callbacks
- **peft**: LoRA/QLoRA adapter management — only adapters are trained, not full weights
- **accelerate**: Handles mixed precision (fp16/bf16) and gradient checkpointing automatically
- **bitsandbytes**: 4-bit NF4 quantization to fit large models into 16 GB VRAM
- **datasets**: HuggingFace datasets with efficient Arrow caching
- **langdetect / ftfy**: Language detection and Unicode text repair for Amharic
- **gspread / oauth2client**: Google Sheets API access

### Expected Output
All packages install without errors. Unsloth prints its version and GPU detection.

### Common Errors
- `CUDA not available`: Runtime is CPU — switch to T4 GPU
- `torch version mismatch`: Restart runtime after install

In [ ]:
# ─────────────────────────────────────────────
# SECTION 1: Install Dependencies
# ─────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[WARN] {result.stderr[:300]}")
    return result

print("[1/6] Installing Unsloth (T4-optimized)...")
run("pip install unsloth[colab-new] -q")

print("[2/6] Installing TRL, PEFT, Accelerate, BitsAndBytes...")
run("pip install trl peft accelerate bitsandbytes -q")

print("[3/6] Installing HuggingFace Datasets & Transformers...")
run("pip install datasets transformers -q")

print("[4/6] Installing Google Sheets connectors...")
run("pip install gspread oauth2client google-auth google-auth-oauthlib -q")

print("[5/6] Installing NLP utilities...")
run("pip install langdetect ftfy unicodedata2 pandas numpy scikit-learn -q")

print("[6/6] Installing evaluation libraries...")
run("pip install rouge-score sacrebleu bert-score -q")

print("\n✅ All dependencies installed.")

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  NO GPU DETECTED — Switch runtime to T4 GPU before continuing!")

---
## 🔗 SECTION 2 — Google Drive Mount & Google Sheets Data Loading

### Design Decision
We read directly from Google Sheets (not CSV export) so the team can update data in place and re-run the notebook. We also immediately checkpoint a local Parquet copy for reproducibility.

### Expected Output
- Drive mounted at `/content/drive`
- DataFrame with columns: `user`, `assistant` (minimum)
- Shape printed: approximately `(53000, N)`

### Common Errors
- `SpreadsheetNotFound`: Wrong SHEET_ID or sheet not shared with service account
- `gspread.exceptions.APIError: RESOURCE_EXHAUSTED`: Too many API calls — add `time.sleep(2)` between batch reads

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2A: Mount Google Drive
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
BASE_DIR = '/content/drive/MyDrive/Riftronix_AgriBot'
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f'{BASE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{BASE_DIR}/adapters', exist_ok=True)
os.makedirs(f'{BASE_DIR}/merged_model', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data', exist_ok=True)
os.makedirs(f'{BASE_DIR}/logs', exist_ok=True)
os.makedirs(f'{BASE_DIR}/evals', exist_ok=True)

print(f"✅ Drive mounted. Project directory: {BASE_DIR}")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2B: Load Dataset from Google Sheets
# ─────────────────────────────────────────────
# ⚙️  CONFIGURE THESE VALUES:
SHEET_ID   = 'YOUR_GOOGLE_SHEET_ID_HERE'   # From the sheet URL
SHEET_NAME = 'Sheet1'                       # Tab name inside the spreadsheet
USER_COL   = 'user'                         # Column header for user messages
ASST_COL   = 'assistant'                    # Column header for assistant messages

import gspread
from google.colab import auth
from google.auth import default as google_default
import pandas as pd
import time

auth.authenticate_user()
creds, _ = google_default()
gc = gspread.authorize(creds)

print(f"Opening sheet: {SHEET_ID} → tab: {SHEET_NAME}")
sh = gc.open_by_key(SHEET_ID)
worksheet = sh.worksheet(SHEET_NAME)

print("Fetching all records (may take 30–90 seconds for 53k rows)...")
records = worksheet.get_all_records()
df_raw = pd.DataFrame(records)

print(f"\n✅ Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

# Save checkpoint immediately
RAW_PATH = f'{BASE_DIR}/data/raw_dataset.parquet'
df_raw.to_parquet(RAW_PATH, index=False)
print(f"\n💾 Raw data saved to: {RAW_PATH}")

---
## ✅ SECTION 3 — Data Validation

### What We Check
1. **Schema**: Required columns exist
2. **Nulls**: Any missing user or assistant turns
3. **Empty strings**: Whitespace-only entries
4. **Language detection**: Confirm majority is Amharic (Ethiopic script)
5. **Length distribution**: Flag extremely short (<5 chars) or long (>2000 chars) entries
6. **Duplicate detection**: Exact and near-duplicate pairs

### Expected Output
A printed validation report and a `validation_report.txt` saved to Drive.

In [ ]:
# ─────────────────────────────────────────────
# SECTION 3: Data Validation
# ─────────────────────────────────────────────
import re
import pandas as pd
import numpy as np
from collections import Counter

df = pd.read_parquet(RAW_PATH).copy()

report_lines = ["=" * 60, "DATA VALIDATION REPORT", "=" * 60]
issues = []

# ── 1. Schema Check ──────────────────────────
for col in [USER_COL, ASST_COL]:
    if col not in df.columns:
        raise ValueError(f"❌ Required column '{col}' not found! Found: {list(df.columns)}")
report_lines.append(f"[SCHEMA]  ✅ Required columns present: {USER_COL}, {ASST_COL}")

# ── 2. Null / Empty Check ─────────────────────
null_user = df[USER_COL].isnull().sum()
null_asst = df[ASST_COL].isnull().sum()
empty_user = (df[USER_COL].astype(str).str.strip() == '').sum()
empty_asst = (df[ASST_COL].astype(str).str.strip() == '').sum()
report_lines.append(f"[NULLS]   user={null_user}, assistant={null_asst}")
report_lines.append(f"[EMPTY]   user={empty_user}, assistant={empty_asst}")
if null_user + null_asst + empty_user + empty_asst > 0:
    issues.append('nulls_or_empty')

# ── 3. Length Distribution ─────────────────────
df['user_len']  = df[USER_COL].astype(str).str.len()
df['asst_len']  = df[ASST_COL].astype(str).str.len()
short_user = (df['user_len'] < 5).sum()
short_asst = (df['asst_len'] < 5).sum()
long_user  = (df['user_len'] > 2000).sum()
long_asst  = (df['asst_len'] > 2000).sum()
report_lines.append(f"[LENGTH]  user  — min:{df['user_len'].min()}, median:{df['user_len'].median():.0f}, max:{df['user_len'].max()}")
report_lines.append(f"[LENGTH]  asst  — min:{df['asst_len'].min()}, median:{df['asst_len'].median():.0f}, max:{df['asst_len'].max()}")
report_lines.append(f"[LENGTH]  short user(<5)={short_user}, long user(>2000)={long_user}")
report_lines.append(f"[LENGTH]  short asst(<5)={short_asst}, long asst(>2000)={long_asst}")

# ── 4. Amharic Script Detection ─────────────────
ETHIOPIC_RE = re.compile(r'[\u1200-\u137F\u1380-\u139F\u2D80-\u2DDF\uAB00-\uAB2F]')
def has_amharic(text):
    return bool(ETHIOPIC_RE.search(str(text)))

amharic_user_pct = df[USER_COL].apply(has_amharic).mean() * 100
amharic_asst_pct = df[ASST_COL].apply(has_amharic).mean() * 100
report_lines.append(f"[LANGUAGE] Amharic script in user turns: {amharic_user_pct:.1f}%")
report_lines.append(f"[LANGUAGE] Amharic script in asst turns: {amharic_asst_pct:.1f}%")
if amharic_user_pct < 50:
    issues.append('low_amharic_ratio')
    report_lines.append("  ⚠️  WARNING: Less than 50% Amharic script in user column!")

# ── 5. Exact Duplicate Detection ────────────────
dup_mask = df.duplicated(subset=[USER_COL, ASST_COL], keep='first')
n_dupes = dup_mask.sum()
report_lines.append(f"[DUPES]   Exact duplicate pairs: {n_dupes:,} ({n_dupes/len(df)*100:.1f}%)")
if n_dupes > 0:
    issues.append('duplicates')

# ── 6. Summary ─────────────────────────────────
report_lines.append("=" * 60)
report_lines.append(f"Total rows:  {len(df):,}")
report_lines.append(f"Issues found: {issues if issues else 'None — dataset looks clean!'}")
report_lines.append("=" * 60)

full_report = "\n".join(report_lines)
print(full_report)

with open(f'{BASE_DIR}/logs/validation_report.txt', 'w', encoding='utf-8') as f:
    f.write(full_report)
print(f"\n💾 Validation report saved.")

---
## 🧹 SECTION 4 — Data Cleaning & Quality Scoring

### Pipeline Steps
1. **Drop nulls / empty rows**
2. **Unicode normalization** (NFC) — critical for Amharic composite characters
3. **Amharic text normalization** — handle common character variants (e.g., ሀ/ሃ/ሐ)
4. **Remove exact duplicates**
5. **Length filtering** — remove extremely short/long pairs
6. **Quality scoring** — assign a score to each pair for curriculum learning
7. **Save cleaned dataset**

In [ ]:
# ─────────────────────────────────────────────
# SECTION 4A: Cleaning — Normalize & Filter
# ─────────────────────────────────────────────
import unicodedata
import ftfy
import re

df = pd.read_parquet(RAW_PATH).copy()

# ── Step 1: Drop nulls and empty ─────────────
before = len(df)
df = df.dropna(subset=[USER_COL, ASST_COL])
df[USER_COL] = df[USER_COL].astype(str).str.strip()
df[ASST_COL] = df[ASST_COL].astype(str).str.strip()
df = df[(df[USER_COL] != '') & (df[ASST_COL] != '')]
print(f"After null/empty drop:  {len(df):,}  (removed {before - len(df):,})")

# ── Step 2: Unicode normalization (NFC) ──────
def normalize_unicode(text):
    text = ftfy.fix_text(text)                   # Fix garbled encoding
    text = unicodedata.normalize('NFC', text)    # Compose Amharic composites
    return text

df[USER_COL] = df[USER_COL].apply(normalize_unicode)
df[ASST_COL] = df[ASST_COL].apply(normalize_unicode)
print("Unicode normalization (NFC + ftfy): done")

# ── Step 3: Amharic character normalization ──
# Normalize common Amharic homophone variants to canonical forms
# These characters are phonetically identical in standard Amharic
AMHARIC_NORM_MAP = {
    # ሀ family → ሀ canonical
    'ሐ': 'ሀ', 'ሑ': 'ሁ', 'ሒ': 'ሂ', 'ሓ': 'ሃ', 'ሔ': 'ሄ', 'ሕ': 'ህ', 'ሖ': 'ሆ',
    # ሠ family → ሰ canonical
    'ሠ': 'ሰ', 'ሡ': 'ሱ', 'ሢ': 'ሲ', 'ሣ': 'ሳ', 'ሤ': 'ሴ', 'ሥ': 'ስ', 'ሦ': 'ሶ',
    # ዐ family → አ canonical
    'ዐ': 'አ', 'ዑ': 'ኡ', 'ዒ': 'ኢ', 'ዓ': 'አ', 'ዔ': 'ኤ', 'ዕ': 'እ', 'ዖ': 'ኦ',
    # ጸ/ፀ normalization
    'ፀ': 'ጸ', 'ፁ': 'ጹ', 'ፂ': 'ጺ', 'ፃ': 'ጻ', 'ፄ': 'ጼ', 'ፅ': 'ጽ', 'ፆ': 'ጾ',
}

def amharic_normalize(text):
    for src, tgt in AMHARIC_NORM_MAP.items():
        text = text.replace(src, tgt)
    # Normalize multiple spaces/newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df[USER_COL] = df[USER_COL].apply(amharic_normalize)
df[ASST_COL] = df[ASST_COL].apply(amharic_normalize)
print("Amharic character normalization: done")

# ── Step 4: Remove exact duplicates ──────────
before = len(df)
df = df.drop_duplicates(subset=[USER_COL, ASST_COL], keep='first')
print(f"After dedup: {len(df):,}  (removed {before - len(df):,} exact duplicates)")

# ── Step 5: Length filtering ─────────────────
df['user_len'] = df[USER_COL].str.len()
df['asst_len'] = df[ASST_COL].str.len()
before = len(df)
df = df[(df['user_len'] >= 5) & (df['asst_len'] >= 10)]
df = df[(df['user_len'] <= 1500) & (df['asst_len'] <= 2500)]
print(f"After length filter: {len(df):,}  (removed {before - len(df):,})")

print(f"\n✅ Cleaned dataset: {len(df):,} rows")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 4B: Quality Scoring
# ─────────────────────────────────────────────
# A composite quality score (0–100) for each pair.
# Used later for curriculum learning (train on high-quality first).

ETHIOPIC_RE = re.compile(r'[\u1200-\u137F\u1380-\u139F]')

# Agri keywords in Amharic and English
AGRI_KEYWORDS_AM = [
    'እርሻ', 'ሰብል', 'አፈር', 'ዘር', 'ምርት', 'ማዳበሪያ', 'መስኖ', 'ተክል',
    'ቆሻሻ', 'ፀረ-ተባይ', 'ምርምር', 'ዘርፍ', 'ደን', 'ሰብሎች', 'ወቅት'
]
AGRI_KEYWORDS_EN = [
    'crop', 'soil', 'fertilizer', 'irrigation', 'harvest', 'seed',
    'pest', 'disease', 'yield', 'farm', 'agriculture', 'plant'
]

def quality_score(row):
    user = row[USER_COL]
    asst = row[ASST_COL]
    score = 0

    # 1. Amharic script presence (40 pts)
    am_ratio = len(ETHIOPIC_RE.findall(asst)) / max(len(asst), 1)
    score += min(40, am_ratio * 100)

    # 2. Agricultural keyword presence (30 pts)
    combined = user + ' ' + asst
    agri_hits = sum(1 for kw in AGRI_KEYWORDS_AM if kw in combined)
    agri_hits += sum(1 for kw in AGRI_KEYWORDS_EN if kw.lower() in combined.lower())
    score += min(30, agri_hits * 5)

    # 3. Answer length adequacy (20 pts) — prefer 50–500 chars
    alen = len(asst)
    if 50 <= alen <= 500:
        score += 20
    elif alen < 50:
        score += max(0, alen * 0.4)
    else:
        score += max(0, 20 - (alen - 500) / 100)

    # 4. Question has ? mark or imperative (10 pts)
    if '?' in user or '፧' in user or user.endswith('ን') or user.endswith('ልቻ'):
        score += 10

    return round(min(score, 100), 2)

print("Computing quality scores (may take ~30s)...")
df['quality_score'] = df.apply(quality_score, axis=1)

print(f"Quality score distribution:")
print(df['quality_score'].describe())
print(f"\nHigh quality (≥60): {(df['quality_score'] >= 60).sum():,}")
print(f"Medium quality (30–59): {((df['quality_score'] >= 30) & (df['quality_score'] < 60)).sum():,}")
print(f"Low quality (<30): {(df['quality_score'] < 30).sum():,}")

# Drop truly low-quality rows
before = len(df)
df = df[df['quality_score'] >= 15].copy()
print(f"\nAfter quality filter (≥15): {len(df):,} rows (removed {before-len(df):,})")

CLEAN_PATH = f'{BASE_DIR}/data/cleaned_dataset.parquet'
df.to_parquet(CLEAN_PATH, index=False)
print(f"\n💾 Cleaned + scored dataset: {CLEAN_PATH}")

---
## 🔤 SECTION 5 — Tokenizer Analysis

### What is Tokenizer Fertility?
Fertility = average tokens per word. For English, well-tuned tokenizers score ~1.2–1.5.  
For Amharic on most multilingual models, fertility is **3–8×** — meaning one Amharic word needs 3–8 tokens. This:
- Wastes context window space
- Increases training cost
- Degrades model quality

### Decision: Should We Extend the Tokenizer?
**Recommendation: NO for initial training run.**  
Tokenizer extension requires re-initializing embedding weights for new tokens, which is risky on T4 with limited VRAM.  
Instead, we choose Qwen2.5 which already has reasonable Amharic coverage.  
Re-evaluate after first training cycle.

### Expected Output
A fertility report comparing models.

In [ ]:
# ─────────────────────────────────────────────
# SECTION 5: Tokenizer Fertility Analysis
# ─────────────────────────────────────────────
from transformers import AutoTokenizer
import numpy as np

# Sample 500 Amharic responses for analysis
df_clean = pd.read_parquet(CLEAN_PATH)
SAMPLE = df_clean[ASST_COL].sample(min(500, len(df_clean)), random_state=42).tolist()

# Models to compare (we'll load tokenizers only — no full model download)
MODELS_TO_ANALYZE = [
    'Qwen/Qwen2.5-3B-Instruct',
    'google/gemma-2-2b-it',
]

def compute_fertility(tokenizer, texts):
    """Tokens per character ratio (lower = more efficient for the language)"""
    ratios = []
    for text in texts[:200]:  # limit for speed
        chars = len(text)
        tokens = len(tokenizer.encode(text, add_special_tokens=False))
        if chars > 0:
            ratios.append(tokens / chars)
    return np.mean(ratios), np.std(ratios)

fertility_report = []
for model_name in MODELS_TO_ANALYZE:
    try:
        print(f"Loading tokenizer: {model_name}")
        tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        vocab_size = tok.vocab_size

        # Check Amharic coverage in vocab
        am_tokens = [v for v in tok.get_vocab().keys()
                     if re.search(r'[\u1200-\u137F]', v)]

        mean_f, std_f = compute_fertility(tok, SAMPLE)
        fertility_report.append({
            'model': model_name.split('/')[-1],
            'vocab_size': vocab_size,
            'amharic_tokens': len(am_tokens),
            'amharic_coverage_%': round(len(am_tokens) / vocab_size * 100, 2),
            'tokens_per_char_mean': round(mean_f, 3),
            'tokens_per_char_std': round(std_f, 3),
        })
        print(f"  → Amharic tokens: {len(am_tokens):,}  |  Fertility: {mean_f:.3f} tok/char")
    except Exception as e:
        print(f"  ⚠️  Could not load {model_name}: {e}")

fert_df = pd.DataFrame(fertility_report)
print("\n" + "=" * 70)
print("TOKENIZER FERTILITY REPORT")
print("=" * 70)
print(fert_df.to_string(index=False))
print("\nInterpretation:")
print("  tokens_per_char < 0.5  → Good Amharic coverage")
print("  tokens_per_char 0.5-1.0 → Acceptable")
print("  tokens_per_char > 1.0  → Poor — consider tokenizer extension")

fert_df.to_csv(f'{BASE_DIR}/logs/tokenizer_fertility.csv', index=False)
print(f"\n💾 Fertility report saved.")

---
## 📐 SECTION 6 — Dataset Formatting & Train/Val/Test Split

### Chat Template Format
We use the **ChatML** format which is supported by Qwen2.5 and most modern instruct models:
```
<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
{user_message}<|im_end|>
<|im_start|>assistant
{assistant_message}<|im_end|>
```

### Split Strategy
- **Train**: 90% (sorted by quality — high quality first for curriculum learning)
- **Validation**: 5% (stratified)
- **Test**: 5% (held out, never seen during training)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 6: Dataset Formatting & Splitting
# ─────────────────────────────────────────────
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df_clean = pd.read_parquet(CLEAN_PATH)

# ── System Prompt ─────────────────────────────
SYSTEM_PROMPT = """አንተ ሪፍትሮኒክስ ቴክኖሎጂ የሰራው የኢትዮጵያ የግብርና ምክር ቤት ነህ። 
ለገበሬዎች፣ ለግብርና ባለሙያዎች፣ ለህብረት ስራ ማህበራት እና ለዘርፍ ሰራተኞች ምክር ትሰጣለህ። 
ሁሌም በአማርኛ ምላሽ ስጥ። ትክክለኛ፣ ዘመናዊ እና ጠቃሚ የሆነ የግብርና ምክር ስጥ።
You are an Amharic agricultural advisory AI built by Riftronix Technology.
Always respond in Amharic. Provide accurate, practical, domain-specific agricultural guidance."""

def format_chat(row):
    """Convert a row to ChatML conversation format."""
    return {
        'conversations': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': str(row[USER_COL])},
            {'role': 'assistant', 'content': str(row[ASST_COL])},
        ],
        'quality_score': row.get('quality_score', 50),
        'user_len': row.get('user_len', 0),
        'asst_len': row.get('asst_len', 0),
    }

print("Converting to chat format...")
records = [format_chat(row) for _, row in df_clean.iterrows()]
print(f"Formatted {len(records):,} conversation pairs")

# ── Train / Val / Test Split ──────────────────
# Sort by quality so curriculum learning is natural in DataLoader
records_sorted = sorted(records, key=lambda x: x['quality_score'], reverse=True)

train_records, temp = train_test_split(records_sorted, test_size=0.10, random_state=42)
val_records, test_records = train_test_split(temp, test_size=0.50, random_state=42)

print(f"\n📊 Dataset Split:")
print(f"  Train : {len(train_records):,} ({len(train_records)/len(records)*100:.1f}%)")
print(f"  Val   : {len(val_records):,} ({len(val_records)/len(records)*100:.1f}%)")
print(f"  Test  : {len(test_records):,} ({len(test_records)/len(records)*100:.1f}%)")

# ── Save as HuggingFace Datasets ──────────────
def to_hf_dataset(records_list):
    return Dataset.from_list(records_list)

dataset_dict = DatasetDict({
    'train': to_hf_dataset(train_records),
    'validation': to_hf_dataset(val_records),
    'test': to_hf_dataset(test_records),
})

DATASET_PATH = f'{BASE_DIR}/data/formatted_dataset'
dataset_dict.save_to_disk(DATASET_PATH)
print(f"\n💾 Dataset saved: {DATASET_PATH}")
print(f"\nSample training conversation:")
print(train_records[0]['conversations'][1]['content'][:200])

---
## 🤖 SECTION 7 — Model Loading with Unsloth (4-bit QLoRA)

### Model Selection: **Qwen2.5-3B-Instruct**

| Criterion | Qwen2.5-3B | Gemma-2-2B | Llama-3.2-3B |
|-----------|------------|------------|---------------|
| Amharic coverage | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| Multilingual | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| T4 fit (4-bit) | ✅ ~3.5 GB | ✅ ~2.5 GB | ✅ ~3.5 GB |
| Instruction following | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| Chat template | ChatML | Gemma | Llama-3 |
| License | Apache 2.0 | Gemma | Llama 3 |

**Winner: Qwen2.5-3B-Instruct** — best Amharic tokenizer coverage, top multilingual benchmarks, Apache 2.0.

### Unsloth Advantages
- 2× faster training via fused attention + custom kernels
- 60% VRAM savings vs standard HuggingFace QLoRA
- Native T4 support
- Drop-in replacement for HuggingFace trainer

### Expected Output
Model loads in ~2–4 minutes. Prints model size and trainable parameters.

In [ ]:
# ─────────────────────────────────────────────
# SECTION 7: Load Model with Unsloth
# ─────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

# ⚙️  CONFIGURATION — adjust if swapping models
MODEL_NAME   = 'Qwen/Qwen2.5-3B-Instruct'  # Base model
MAX_SEQ_LEN  = 2048    # Max input+output tokens. Amharic is verbose — 2048 is safe on T4
LOAD_IN_4BIT = True    # 4-bit NF4 quantization for T4 16GB
DTYPE        = None    # None = auto-detect (fp16 on T4)

print(f"Loading: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LEN}")
print(f"4-bit quantization: {LOAD_IN_4BIT}")
print("This takes 2–5 minutes on first download...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name         = MODEL_NAME,
    max_seq_length     = MAX_SEQ_LEN,
    dtype              = DTYPE,
    load_in_4bit       = LOAD_IN_4BIT,
    trust_remote_code  = True,
)

# Confirm tokenizer has EOS token (needed for SFTTrainer packing)
if tokenizer.eos_token is None:
    tokenizer.eos_token = '<|im_end|>'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # Required for training stability

# Memory report
gpu_mem = torch.cuda.memory_allocated() / 1e9
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n✅ Model loaded successfully!")
print(f"GPU memory used: {gpu_mem:.1f} / {gpu_total:.1f} GB")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

---
## ⚙️ SECTION 8 — QLoRA Configuration (PEFT)

### Why These Hyperparameters?

| Parameter | Value | Justification |
|-----------|-------|---------------|
| `lora_r` | 32 | Rank 32: good balance of expressiveness vs memory on T4 |
| `lora_alpha` | 64 | alpha = 2×r is the standard rule for stable training |
| `lora_dropout` | 0.05 | Low dropout: we want the model to learn the domain |
| Target modules | q,k,v,o,gate,up,down | Full attention + FFN targets for best domain adaptation |
| `use_rslora` | True | Rank-stabilized LoRA — better stability than standard LoRA |
| `use_gradient_checkpointing` | 'unsloth' | Unsloth's custom checkpointing: 30% memory saving |

### Expected Output
Trainable parameters: ~5–10% of total (the LoRA adapters only).

In [ ]:
# ─────────────────────────────────────────────
# SECTION 8: QLoRA / PEFT Configuration
# ─────────────────────────────────────────────
from unsloth import FastLanguageModel

# LoRA target modules for Qwen2.5
# Including FFN (gate_proj, up_proj, down_proj) is critical for
# agricultural domain knowledge injection, not just style.
TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',   # Attention
    'gate_proj', 'up_proj', 'down_proj'         # FFN
]

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 32,       # LoRA rank
    target_modules             = TARGET_MODULES,
    lora_alpha                 = 64,       # Scaling factor: 2×r
    lora_dropout               = 0.05,     # Small dropout for domain learning
    bias                       = 'none',   # No bias for efficiency
    use_gradient_checkpointing = 'unsloth', # Unsloth custom checkpointing
    random_state               = 42,
    use_rslora                 = True,     # Rank-stabilized LoRA
    loftq_config               = None,
)

# Count trainable parameters
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
total      = sum(p.numel() for p in model.parameters())
pct        = 100 * trainable / total

print(f"✅ QLoRA configured:")
print(f"   Trainable params : {trainable:,} ({pct:.2f}%)")
print(f"   Total params     : {total:,}")
print(f"   Frozen params    : {total - trainable:,}")

gpu_mem = torch.cuda.memory_allocated() / 1e9
print(f"\n   GPU memory now   : {gpu_mem:.1f} GB")
print("\n✅ Model ready for fine-tuning!")

---
## 🎛️ SECTION 9 — Trainer Configuration (SFTTrainer + TRL)

### Training Strategy: Supervised Fine-Tuning (SFT)

**Why SFT first?**
- We have 53k labeled conversation pairs — perfect for SFT
- SFT is stable, reproducible, and well-understood
- DPO/RLHF require preference data (we don't have it yet)
- Continued pretraining would need a much larger unlabeled corpus

**Avoid RLHF for now:** Requires reward model training, instability risk, and much more compute.

### Key Hyperparameter Decisions

| Param | Value | Reason |
|-------|-------|--------|
| `learning_rate` | 2e-4 | Standard for QLoRA; with `cosine` decay |
| `per_device_batch_size` | 2 | T4 VRAM limit with 2048 seq len |
| `gradient_accumulation` | 8 | Effective batch = 16 |
| `num_epochs` | 3 | Balanced for 53k pairs; monitor val loss |
| `warmup_ratio` | 0.05 | Stable warmup before peak LR |
| `weight_decay` | 0.01 | Light regularization |
| `packing` | True | Pack multiple short examples → faster training |
| `fp16` | True | T4 doesn't support bf16 natively |

In [ ]:
# ─────────────────────────────────────────────
# SECTION 9: Trainer Configuration
# ─────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_from_disk
from unsloth.chat_templates import get_chat_template

# Load formatted dataset
dataset = load_from_disk(f'{BASE_DIR}/data/formatted_dataset')
print(f"Train: {len(dataset['train']):,} | Val: {len(dataset['validation']):,} | Test: {len(dataset['test']):,}")

# Apply Qwen2.5 ChatML template to tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'qwen-2.5',
)

def formatting_prompts_func(examples):
    """Convert conversations to tokenizer's expected format string."""
    convos = examples['conversations']
    texts  = [tokenizer.apply_chat_template(
                  convo,
                  tokenize=False,
                  add_generation_prompt=False
              ) for convo in convos]
    return {'text': texts}

train_ds = dataset['train'].map(formatting_prompts_func, batched=True)
val_ds   = dataset['validation'].map(formatting_prompts_func, batched=True)
print("✅ Chat template applied to datasets")
print(f"\nSample formatted text (first 400 chars):")
print(train_ds[0]['text'][:400])

# ── Training Arguments ────────────────────────
training_args = TrainingArguments(
    output_dir                 = f'{BASE_DIR}/checkpoints',
    num_train_epochs           = 3,
    per_device_train_batch_size= 2,
    per_device_eval_batch_size = 2,
    gradient_accumulation_steps= 8,       # Effective batch = 16
    warmup_ratio               = 0.05,
    learning_rate              = 2e-4,
    weight_decay               = 0.01,
    lr_scheduler_type          = 'cosine',
    fp16                       = not torch.cuda.is_bf16_supported(),
    bf16                       = torch.cuda.is_bf16_supported(),
    logging_steps              = 20,
    eval_strategy              = 'steps',
    eval_steps                 = 100,
    save_strategy              = 'steps',
    save_steps                 = 200,
    save_total_limit           = 3,       # Keep last 3 checkpoints
    load_best_model_at_end     = True,
    metric_for_best_model      = 'eval_loss',
    greater_is_better          = False,
    report_to                  = 'none',  # Set to 'wandb' if you have a wandb account
    seed                       = 42,
    dataloader_num_workers     = 2,
    optim                      = 'adamw_8bit',  # 8-bit Adam for memory efficiency
    gradient_checkpointing     = True,
    group_by_length            = True,    # Group similar-length sequences → less padding
)

# ── SFT Trainer ───────────────────────────────
trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_ds,
    eval_dataset      = val_ds,
    dataset_text_field= 'text',
    max_seq_length    = MAX_SEQ_LEN,
    dataset_num_proc  = 2,
    packing           = True,   # Pack short sequences for efficiency
    args              = training_args,
)

# Memory summary
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"\n✅ Trainer configured")
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"Reserved memory = {start_gpu_memory} GB.")
print(f"Effective batch size = {2 * 8} (batch × accumulation)")

---
## 🚀 SECTION 10 — Fine-Tuning

### What Happens Here
- The trainer iterates over the training set for 3 epochs
- Every 20 steps: training loss is logged
- Every 100 steps: validation loss is computed
- Every 200 steps: checkpoint saved to Drive
- Best checkpoint (lowest val loss) is automatically reloaded at end

### Estimated Training Time on T4
~53,000 pairs × 3 epochs ÷ effective batch 16 ≈ ~10,000 steps  
At ~0.3s/step → **~50–90 minutes** total

### Warning Signs
- `loss=nan` after step 1: LR too high — reduce to 1e-4
- `CUDA OOM`: Reduce batch size to 1, increase gradient_accumulation to 16
- Loss not decreasing: Check data formatting, ensure labels are not masked

In [ ]:
# ─────────────────────────────────────────────
# SECTION 10: Fine-Tuning
# ─────────────────────────────────────────────
import time

print("🚀 Starting fine-tuning...")
print("   Monitor: loss should decrease from ~2.5 to ~0.8–1.2 over 3 epochs")
print("   Checkpoints are saved to Drive every 200 steps")
print("=" * 60)

t0 = time.time()
train_result = trainer.train()
t1 = time.time()

# ── Training Summary ─────────────────────────
elapsed = t1 - t0
metrics = train_result.metrics

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Time elapsed      : {elapsed/60:.1f} minutes")
print(f"Total steps       : {metrics.get('train_steps_per_second', 0):.0f} steps")
print(f"Final train loss  : {metrics.get('train_loss', 'N/A')}")
print(f"Samples/second    : {metrics.get('train_samples_per_second', 'N/A')}")

used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"Peak GPU memory   : {used_memory} GB")

# Save training metrics
trainer.save_metrics('train', metrics)
import json
with open(f'{BASE_DIR}/logs/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n💾 Metrics saved to Drive.")

---
## 📊 SECTION 11 — Evaluation Framework

### Metrics
1. **Perplexity** — lower = better language model fit
2. **ROUGE-L** — response overlap with reference answers
3. **chrF** — character-level F-score (better than BLEU for Amharic morphology)
4. **BERTScore** — semantic similarity using multilingual encoder
5. **Agricultural relevance** — keyword recall
6. **Response length ratio** — measures verbosity alignment
7. **Amharic output rate** — % of responses in Amharic
8. **Human-style score** — auto-scored via Claude API call (optional)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 11A: Automated Evaluation
# ─────────────────────────────────────────────
from unsloth import FastLanguageModel
from rouge_score import rouge_scorer
from sacrebleu.metrics import CHRF
import torch
import json

# Enable inference mode
FastLanguageModel.for_inference(model)

# Load test set
from datasets import load_from_disk
test_ds = load_from_disk(f'{BASE_DIR}/data/formatted_dataset')['test']

EVAL_SAMPLES = min(200, len(test_ds))  # Evaluate on 200 samples
test_sample  = test_ds.select(range(EVAL_SAMPLES))

chrf_metric  = CHRF()
rouge        = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

ETHIOPIC_RE  = re.compile(r'[\u1200-\u137F]')

results = []
print(f"Evaluating on {EVAL_SAMPLES} test samples...")

for i, example in enumerate(test_sample):
    convos = example['conversations']
    user_msg   = convos[1]['content']   # user turn
    reference  = convos[2]['content']   # expected answer

    # Build prompt (system + user, no assistant turn)
    prompt_convos = [convos[0], convos[1]]
    prompt_text   = tokenizer.apply_chat_template(
        prompt_convos, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt_text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens   = 400,
            temperature      = 0.3,
            do_sample        = True,
            top_p            = 0.9,
            repetition_penalty = 1.1,
            pad_token_id     = tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Compute metrics
    rouge_score  = rouge.score(reference, generated)['rougeL'].fmeasure
    chrf_score   = chrf_metric.sentence_score(generated, [reference]).score / 100
    am_output    = len(ETHIOPIC_RE.findall(generated)) / max(len(generated), 1)

    results.append({
        'user': user_msg[:100],
        'reference': reference[:100],
        'generated': generated[:100],
        'rouge_l': round(rouge_score, 4),
        'chrf': round(chrf_score, 4),
        'amharic_output_rate': round(am_output, 3),
        'gen_len': len(generated),
    })

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{EVAL_SAMPLES} done...")

# ── Aggregate Results ─────────────────────────
results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"ROUGE-L         : {results_df['rouge_l'].mean():.4f} ± {results_df['rouge_l'].std():.4f}")
print(f"chrF            : {results_df['chrf'].mean():.4f} ± {results_df['chrf'].std():.4f}")
print(f"Amharic output% : {results_df['amharic_output_rate'].mean()*100:.1f}%")
print(f"Avg gen length  : {results_df['gen_len'].mean():.0f} chars")

results_df.to_csv(f'{BASE_DIR}/evals/eval_results.csv', index=False)
print(f"\n💾 Results saved to Drive.")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 11B: Interactive Inference Test
# ─────────────────────────────────────────────
def chat_amharic(user_message, max_new_tokens=400, temperature=0.3):
    """Run inference with the fine-tuned model."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_message},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            do_sample          = True,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    return response

# Test with sample agricultural questions
test_questions = [
    'በቆሎ ለማምረት ምን ዓይነት አፈር ያስፈልጋል?',          # What soil is needed for maize?
    'ቲማቲም ተክሉ ምን ዓይነት በሽታ ሊያጠቃ ይችላል?',       # What diseases can affect tomato plants?
    'ለትንሽ ገበሬ ተስማሚ የሆነ የመስኖ ዘዴ ምን ነው?',       # What irrigation method suits small farmers?
]

print("\n🌾 INFERENCE TEST — Fine-tuned Amharic AgriBot\n")
print("=" * 60)
for q in test_questions:
    print(f"\n👤 User: {q}")
    answer = chat_amharic(q)
    print(f"🤖 Bot : {answer[:500]}")
    print("-" * 60)

---
##  SECTION 12 — Benchmarking Framework

### Benchmark Design
Compare configurations across:
1. **LoRA rank** (r=16 vs r=32 vs r=64)
2. **Learning rate** (1e-4 vs 2e-4 vs 5e-4)
3. **Sequence length** (1024 vs 2048)

Each config is evaluated on the same 200-sample test set.

In [ ]:
# ─────────────────────────────────────────────
# SECTION 12: Benchmarking Results Tracker
# ─────────────────────────────────────────────
# This cell records and displays benchmark results.
# Run different configurations and fill in the table.

import pandas as pd

# Pre-filled example benchmark table
# Fill in your actual results after each experiment run
benchmark_data = [
    {
        'experiment': 'Baseline (r=32, lr=2e-4, seq=2048)',
        'model': 'Qwen2.5-3B-Instruct',
        'lora_r': 32, 'lora_alpha': 64,
        'learning_rate': '2e-4', 'seq_len': 2048,
        'epochs': 3, 'batch_eff': 16,
        'train_loss': '—',    # Fill after run
        'val_loss': '—',
        'rouge_l': '—',
        'chrf': '—',
        'amharic_pct': '—',
        'train_time_min': '—',
        'peak_vram_gb': '—',
        'notes': 'Main recommended config',
    },
    {
        'experiment': 'Low rank (r=16, lr=2e-4)',
        'model': 'Qwen2.5-3B-Instruct',
        'lora_r': 16, 'lora_alpha': 32,
        'learning_rate': '2e-4', 'seq_len': 2048,
        'epochs': 3, 'batch_eff': 16,
        'train_loss': '—', 'val_loss': '—',
        'rouge_l': '—', 'chrf': '—',
        'amharic_pct': '—', 'train_time_min': '—',
        'peak_vram_gb': '—',
        'notes': 'Faster but less expressive',
    },
    {
        'experiment': 'High LR (r=32, lr=5e-4)',
        'model': 'Qwen2.5-3B-Instruct',
        'lora_r': 32, 'lora_alpha': 64,
        'learning_rate': '5e-4', 'seq_len': 2048,
        'epochs': 3, 'batch_eff': 16,
        'train_loss': '—', 'val_loss': '—',
        'rouge_l': '—', 'chrf': '—',
        'amharic_pct': '—', 'train_time_min': '—',
        'peak_vram_gb': '—',
        'notes': 'Risk of instability',
    },
    {
        'experiment': 'Short context (r=32, seq=1024)',
        'model': 'Qwen2.5-3B-Instruct',
        'lora_r': 32, 'lora_alpha': 64,
        'learning_rate': '2e-4', 'seq_len': 1024,
        'epochs': 3, 'batch_eff': 16,
        'train_loss': '—', 'val_loss': '—',
        'rouge_l': '—', 'chrf': '—',
        'amharic_pct': '—', 'train_time_min': '—',
        'peak_vram_gb': '—',
        'notes': 'Faster, may truncate long answers',
    },
]

bench_df = pd.DataFrame(benchmark_data)
print("📊 BENCHMARK TABLE")
print("=" * 80)
print(bench_df[['experiment', 'lora_r', 'learning_rate', 'seq_len',
                'train_loss', 'val_loss', 'rouge_l', 'chrf', 'amharic_pct',
                'train_time_min']].to_string(index=False))

bench_df.to_csv(f'{BASE_DIR}/logs/benchmark_results.csv', index=False)
print(f"\n💾 Benchmark table saved.")
print("\n📝 Instructions: Run each config and fill in the '—' values above.")

---
## 💾 SECTION 13 — Save Model, Adapters & Export

### What We Save
1. **LoRA adapters** — lightweight (~50–200 MB), shareable, re-mergeable
2. **Merged model** (16-bit) — full model for deployment
3. **GGUF (Q4_K_M)** — quantized for CPU/Ollama deployment
4. **Tokenizer** — always save alongside the model

### Expected Output
All artifacts in `/content/drive/MyDrive/Riftronix_AgriBot/`

In [ ]:
# ─────────────────────────────────────────────
# SECTION 13A: Save LoRA Adapters
# ─────────────────────────────────────────────
ADAPTER_PATH = f'{BASE_DIR}/adapters'

print("Saving LoRA adapters (lightweight, ~100MB)...")
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"✅ Adapters saved: {ADAPTER_PATH}")

# Save adapter config info
adapter_info = {
    'base_model': MODEL_NAME,
    'lora_r': 32,
    'lora_alpha': 64,
    'target_modules': TARGET_MODULES,
    'max_seq_length': MAX_SEQ_LEN,
    'training_rows': len(train_ds),
    'dataset_source': f'Google Sheets: {SHEET_ID}',
}
with open(f'{ADAPTER_PATH}/adapter_config_info.json', 'w') as f:
    json.dump(adapter_info, f, indent=2, ensure_ascii=False)
print("✅ Adapter config info saved.")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 13B: Save Merged Model (16-bit)
# ─────────────────────────────────────────────
# The merged model combines base weights + LoRA adapters.
# Use this for production deployment.
# WARNING: This takes ~10 min and requires ~10 GB Drive space.

MERGED_PATH = f'{BASE_DIR}/merged_model'

print("Merging and saving 16-bit model (~10 min, ~8 GB)...")
print("⚠️  Ensure Drive has at least 10 GB free.")

model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method = 'merged_16bit',
)
print(f"\n✅ Merged 16-bit model saved: {MERGED_PATH}")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 13C: Export GGUF (for CPU / Ollama deployment)
# ─────────────────────────────────────────────
# Q4_K_M quantization: best quality/size tradeoff
# Final GGUF file is ~2 GB — deployable on any CPU

GGUF_PATH = f'{BASE_DIR}/gguf'
import os
os.makedirs(GGUF_PATH, exist_ok=True)

print("Exporting Q4_K_M GGUF (~15–20 min)...")
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method = 'q4_k_m',
)
print(f"\n✅ GGUF model saved: {GGUF_PATH}")
print("   Deploy with Ollama: ollama create riftronix-agribot -f Modelfile")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 13D: Artifact Summary
# ─────────────────────────────────────────────
import os

print("\n" + "=" * 60)
print("📦 EXPORTED ARTIFACTS SUMMARY")
print("=" * 60)

paths_to_check = [
    (f'{BASE_DIR}/adapters',       'LoRA Adapters (reload for further training)'),
    (f'{BASE_DIR}/merged_model',   'Merged 16-bit Model (production HF deployment)'),
    (f'{BASE_DIR}/gguf',           'GGUF Q4_K_M (Ollama / CPU inference)'),
    (f'{BASE_DIR}/data',           'Processed Datasets'),
    (f'{BASE_DIR}/logs',           'Training Logs & Reports'),
    (f'{BASE_DIR}/evals',          'Evaluation Results'),
    (f'{BASE_DIR}/checkpoints',    'Training Checkpoints'),
]

for path, desc in paths_to_check:
    if os.path.exists(path):
        size = sum(os.path.getsize(os.path.join(dirpath, f))
                   for dirpath, _, files in os.walk(path)
                   for f in files) / 1e9
        print(f"  ✅ {desc}")
        print(f"     Path: {path}")
        print(f"     Size: {size:.2f} GB")
    else:
        print(f"  ⬜ {desc} — not generated yet")
        print(f"     Path: {path}")

print("\n✅ All artifacts are in your Google Drive.")
print("   Access at: drive.google.com → MyDrive → Riftronix_AgriBot")

---
## 🛠️ SECTION 14 — Troubleshooting Guide

### Common Issues & Fixes

| Problem | Cause | Fix |
|---------|-------|-----|
| `CUDA out of memory` | VRAM exceeded | Reduce `per_device_train_batch_size` to 1, increase `gradient_accumulation_steps` to 16 |
| `loss=nan` at step 1 | LR too high or bad data | Reduce LR to `5e-5`; check for empty strings in dataset |
| Training extremely slow | packing=False + long sequences | Set `packing=True` |
| Model outputs English | System prompt not applied | Verify `SYSTEM_PROMPT` is in Amharic and is being injected |
| `SpreadsheetNotFound` | Wrong Sheet ID | Check URL: `docs.google.com/spreadsheets/d/**SHEET_ID**/` |
| Colab disconnects | Long training session | Use Colab Pro or save checkpoints every 100 steps |
| Low ROUGE score | Data quality issue | Filter to quality_score ≥ 50 and retrain |
| Repetitive output | Temperature too low | Increase temperature to 0.5–0.7 |
| `ModuleNotFoundError: unsloth` | Install order | Restart runtime after Section 1 install |

In [ ]:
# ─────────────────────────────────────────────
# SECTION 14: Diagnostic Utilities
# ─────────────────────────────────────────────

def run_diagnostics():
    """Print a full environment diagnostic — run this if anything breaks."""
    import torch, sys, subprocess

    print("=" * 60)
    print("ENVIRONMENT DIAGNOSTICS")
    print("=" * 60)
    print(f"Python       : {sys.version}")
    print(f"PyTorch      : {torch.__version__}")
    print(f"CUDA         : {torch.version.cuda}")
    print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

    if torch.cuda.is_available():
        total   = torch.cuda.get_device_properties(0).total_memory / 1e9
        alloc   = torch.cuda.memory_allocated() / 1e9
        reserved= torch.cuda.memory_reserved() / 1e9
        free    = total - reserved
        print(f"VRAM total   : {total:.1f} GB")
        print(f"VRAM allocated: {alloc:.1f} GB")
        print(f"VRAM reserved: {reserved:.1f} GB")
        print(f"VRAM free    : {free:.1f} GB")

    # Disk space
    result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
    print(f"\nDisk (Colab):\n{result.stdout}")

    # Drive space
    result2 = subprocess.run(['df', '-h', '/content/drive'], capture_output=True, text=True)
    print(f"Drive:\n{result2.stdout}")

    # Key packages
    for pkg in ['unsloth', 'trl', 'peft', 'transformers', 'accelerate', 'bitsandbytes']:
        try:
            mod = __import__(pkg)
            ver = getattr(mod, '__version__', 'unknown')
            print(f"{pkg:20} : {ver}")
        except ImportError:
            print(f"{pkg:20} : ❌ NOT INSTALLED")

    print("=" * 60)

# Run now
run_diagnostics()

In [ ]:
# ─────────────────────────────────────────────
# SECTION 14B: OOM Emergency Recovery
# Run this if you get CUDA OOM error
# ─────────────────────────────────────────────
import torch, gc

def clear_gpu_memory():
    """Aggressively free GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free = (torch.cuda.get_device_properties(0).total_memory -
            torch.cuda.memory_reserved()) / 1e9
    print(f"✅ GPU cache cleared. Free VRAM: {free:.1f} GB")

# Reduce batch config for OOM situations
OOM_RECOVERY_ARGS = {
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 16,  # Keep effective batch = 16
    'max_seq_length': 1024,             # Halve sequence length
    'gradient_checkpointing': True,
}

clear_gpu_memory()
print("\n🔧 OOM Recovery Config:")
for k, v in OOM_RECOVERY_ARGS.items():
    print(f"  {k}: {v}")
print("\nApply these values in Section 9 TrainingArguments and re-run.")

---
## 📋 Final Recommendation Summary

| Category | Recommendation |
|----------|---------------|
| **Base Model** | `Qwen/Qwen2.5-3B-Instruct` |
| **Training Method** | Supervised Fine-Tuning (SFT) with QLoRA |
| **LoRA rank** | r=32, alpha=64 |
| **Learning rate** | 2e-4 with cosine decay |
| **Sequence length** | 2048 |
| **Epochs** | 3 (monitor val loss, stop early if plateaus) |
| **Effective batch** | 16 (batch=2, accumulation=8) |
| **Target modules** | q,k,v,o,gate,up,down projections |
| **Tokenizer extension** | No (revisit after first run) |
| **Evaluation** | chrF + ROUGE-L + Amharic output rate |
| **Deployment** | GGUF Q4_K_M via Ollama for low-cost inference |

### Next Steps After Training
1. Review eval results — target chrF > 0.35, Amharic output > 90%
2. Run DPO with preferred/rejected pairs if quality still needs improvement
3. Consider tokenizer extension experiment if Amharic fertility > 1.0 tok/char
4. Deploy GGUF to AgriTech platform backend

---
*Generated for Riftronix Technology by yeabsra Andnet | Amharic AgriTech Chatbot Project*